# Single-cEll Aggregation for High Resolution Cell States (SEACell) across MCL single-cell RNA sequencing datasets

The following script was run on the individual sample level for each single-cell transcriptome of MCL tumors profiled.

Load the packages.

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import anndata
import os

import SEACells

import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

Define the paths.

In [ ]:
file_path = '/data/scMCL/samples/sample18.h5ad'
sample_name = 'sample18'
plot_out_dir = '/data/scMCL/seacell_out'
data_out_dir = '/data/scMCL/seacell_out'

Load and preprocess the dataset.

In [ ]:
ad = anndata.read_h5ad(file_path)

raw_ad = sc.AnnData(ad.X)
raw_ad.obs_names, raw_ad.var_names = ad.obs_names, ad.var_names
ad.raw = raw_ad
n_cells = ad.n_obs

Process the dataset.

In [ ]:
sc.pp.log1p(ad)
sc.pp.highly_variable_genes(ad, n_top_genes=1500)
sc.tl.pca(ad, n_comps=30, use_highly_variable=True)
sc.pp.neighbors(ad)
sc.tl.umap(ad)

Define the core parameters.

In [ ]:
n_SEACells = int(n_cells / 75)  
build_kernel_on = 'X_pca' 
n_waypoint_eigs = 10 

Initiate the modeling process.

In [ ]:
model = SEACells.core.SEACells(ad, 
                  build_kernel_on=build_kernel_on, 
                  n_SEACells=n_SEACells, 
                  n_waypoint_eigs=n_waypoint_eigs,
                  convergence_epsilon = 1e-5)

Construct the kernel matrix.

In [ ]:
model.construct_kernel_matrix()
M = model.kernel_matrix

Initialize the archetypes.

In [ ]:
model.initialize_archetypes()

Fit the model.

In [ ]:
model.fit(min_iter=10, max_iter=100)

Plot and save model convergence.

In [ ]:
model.plot_convergence(show=False, 
                    save_as=os.path.join(plot_out_dir, f"convergence_plot_{sample_name}.png"))

Export the SEACell assignment.

In [ ]:
ad.obs.to_csv(os.path.join(data_out_dir, f"seacell_output_{sample_name}.csv"))

Calculate the celltype purity per SEACell metacell and save the plot.

In [ ]:
SEACell_purity = SEACells.evaluate.compute_celltype_purity(ad, 'ct_detail_final')
    plt.figure(figsize=(4,4))
    sns.boxplot(data=SEACell_purity, y='ct_detail_final_purity')
    plt.title(f"metacell_purity_{sample_name}")
    sns.despine()
    filename = os.path.join(plot_out_dir, f"metacell_purity_{sample_name}.png")
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()